In [1]:
from cracknuts.acquisition import Acquisition
from cracknuts.acquisition.acquisition import AcqProgress
from enum import Enum
import random
import numpy as np

# %matplotlib ipympl
# import matplotlib.pyplot as plt
sample_length = 1024 * 20

class AlgType(Enum):
    AES_ENC = 1
    AES_DEC = 2
    DES_ENC = 3
    DES_DEC = 4

cmd_set_aes_enc_key = "01 00 00 00 00 00 00 10"
cmd_set_aes_dec_key = "01 01 00 00 00 00 00 10"
cmd_aes_enc_key = "01 02 00 00 00 00 00 10"
cmd_aes_dec_key = "01 03 00 00 00 00 00 10"
key_aes = "11 22 33 44 55 66 77 88 99 00 aa bb cc dd ee ff"
aes_data_len = 16

cmd_set_des_enc_key = "02 00 00 00 00 00 00 08"
cmd_set_des_dec_key = "02 01 00 00 00 00 00 08"
cmd_des_enc_key = "02 02 00 00 00 00 00 08"
cmd_des_dec_key = "02 03 00 00 00 00 00 08"
key_des = "12 34 56 78 90 ab cd ef"
des_data_len = 8

alg = AlgType.AES_ENC # 修改这个值改变算法

class SimpleTestAcq(Acquisition):
    
    # def pre_do(self):
    #     cracker.scrat_arm()

    def init(self):        
        
        cracker.nut_voltage(3300) # 设置NUT工作电压
        cracker.nut_enable(1) # 使能NUT供电
        
        cracker.osc_set_analog_gain(1, 80) # 设置模拟信号增益       
        sample_length = 1024 * 20
        cracker.osc_set_sample_len(sample_length) # 设置采样长度
        cracker.osc_set_sample_delay(1800)

        if alg is AlgType.AES_ENC:
            cmd = cmd_set_aes_enc_key + key_aes
        elif alg is AlgType.AES_DEC:
            cmd = cmd_set_aes_dec_key + key_aes
        elif alg is AlgType.DES_ENC:
            cmd = cmd_set_des_enc_key + key_des
        else:
            cmd = cmd_set_des_dec_key + key_des

        cmd = cmd.replace(' ', '')
        cmd = bytes.fromhex(cmd)
        
        cracker.cracker_serial_data(6, cmd)

    def do(self):
        if alg is AlgType.AES_ENC or AlgType.AES_DEC:
            data = random.randbytes(aes_data_len)
        else:
            data = random.randbytes(des_data_len)
        
        if alg is AlgType.AES_ENC:
            cmd = cmd_aes_enc_key
        elif alg is AlgType.AES_DEC:
            cmd = cmd_aes_dec_key
        elif alg is AlgType.DES_ENC:
            cmd = cmd_des_enc_key
        else:
            cmd = cmd_des_dec_key
        
        cmd = cmd.replace(' ', '')
        cmd = bytes.fromhex(cmd)

        d = cmd + data

        if alg is AlgType.AES_ENC or AlgType.AES_DEC:
            [status, ret] = cracker.cracker_serial_data(6 + aes_data_len, d)
            return data + ret[-aes_data_len:]
        else:
            [status, ret] = cracker.cracker_serial_data(6 + des_data_len, d)
            return data + ret[-des_data_len:]
        
        


In [2]:
from cracknuts.cracker.mock_cracker import MockCracker
from cracknuts.cracker.stateful_cracker import StatefulCracker
from cracknuts.cracker.basic_cracker import CrackerS1
from cracknuts.acquisition import Acquisition as template

# 创建 mock cracker 
# cracker = MockCracker()
cracker = CrackerS1()
# 创建 stateful cracker
cracker = StatefulCracker(cracker)


In [3]:
cracker.set_uri('cnp://192.168.0.10:8080')
cracker.connect()

In [4]:
# cracker.disconnect()

In [5]:
name = cracker.get_name()
cid = cracker.get_id()
version = cracker.get_version()
print(name)
print(cid)
print(version)

Cracker-S1 
0001
0.0.1 


## 采集曲线

In [1]:
from cracknuts_panel import display_cracknuts_panel

acq = SimpleTestAcq(cracker, sample_length = sample_length, data_length=6)
cp = display_cracknuts_panel(acq)
cp

NameError: name 'SimpleTestAcq' is not defined

In [7]:
acq.run(100, sample_length=sample_length, data_length=32)

Exception in thread Thread-5 (_do_run):
Traceback (most recent call last):
  File "d:\anaconda3\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Users\lwj\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "d:\anaconda3\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "E:\codes\cracknuts\src\cracknuts\acquisition\acquisition.py", line 292, in _do_run
    self._loop(not test, self.file_path, self.file_format)
  File "E:\codes\cracknuts\src\cracknuts\acquisition\acquisition.py", line 391, in _loop
    dataset.set_trace(0, trace_index, self._last_wave[1], data)
  File "E:\codes\cracknuts\src\cracknuts\solver\trace.py", line 229, in set_trace
    self._get_under_root(channel_index, self._ARRAY_DATA_PATH)[trace_index] = data
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "d:\anaconda3\Lib\site-packages\zar

In [7]:
# acq.run(10000, sample_length=sample_length, data_length=32 ,file_format='npy')

In [ ]:
acq.stop()

In [6]:
acq.test()

[ERROR] 2024-10-10 20:18:06,510 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,512 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,514 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,515 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,516 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,517 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,518 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,519 cracker.send_and_receive:343 Cracker is not connected.
[ERROR] 2024-10-10 20:18:06,519 acquisition._loop:321 Do error: ("'NoneType' object is not subscriptable",)
[ERROR] 2024-10-10 20:18:06,520 acquisition._loop:324 Exit with do error
[ERROR] 2024-10-10 20:18:06,521 cracker.send_and_receive:343 Cracker is not connected.
Exception in thread Thread-6 (_do_ru

In [7]:
sample_length = 1024 * 18
cracker.osc_set_sample_len(sample_length)
acq.sample_length = sample_length

In [12]:
cracker.osc_set_analog_gain(1, 60)

(0, None)

In [16]:
cracker.nut_voltage(3000)

(0, None)

In [14]:
cracker.osc_set_sample_delay(1800)

(0, None)

In [ ]:
cracker.disconnect()

In [ ]:
acq.sample_length = 1024*10

## 曲线显示

In [19]:
import cracknuts_panel as panel
import numpy as np
import struct
# 打开 trace_dataset 标准格式
from cracknuts.solver.trace import TraceDataset

fixed_trace_dataset = TraceDataset()
random_trace_dataset = TraceDataset()

In [20]:
fixed_tap = panel.display_trace_analysis_panel()
fixed_tap

TraceAnalysisPanelWidget()

In [21]:
random_tap = panel.display_trace_analysis_panel()
random_tap

TraceAnalysisPanelWidget()

In [22]:
folder_name = 'fixed_input'
folder_name = '20241009214129'
# 打开numpy文件
fixed_trace_dataset.load_from_numpy_data_file('dataset//'+folder_name+'/trace.npy')

fixed_tap.set_trace_dataset(fixed_trace_dataset)
fixed_tap.set_show_trace_range(0,9)

In [ ]:
folder_name = '20241009002250'
# folder_name = 'random_input'
# 打开numpy文件
random_trace_dataset.load_from_numpy_data_file('dataset//'+folder_name+'/trace.npy')

random_tap.set_trace_dataset(random_trace_dataset)
random_tap.set_show_trace_range(0,10)

In [8]:
data = np.load('dataset//20241009214129//data.npy')
print(data)

[b'\x12\x1D\xB3\x4B\x5B\x13\x65\xAC\x22\x30\x24\x93\x92\xFF\x5B\x16\x00\x00\x00\x00\x00\x10\x25\x97\xF8\x34\x18\x26\x37\xBF\xA6\x34'
 b'\x22\x98\xD8\xE6\xDB\x78\x84\x79\xA6\x98\x3D\x9A\x90\xAD\x61\x7F\x00\x00\x00\x00\x00\x10\x92\x04\xFB\xF2\x01\x56\x80\x87\x60\x0C'
 b'\xFE\x16\x08\x2D\x23\xA5\x0E\x71\x74\xE6\x49\x78\xE4\xFF\xDC\xF2\x00\x00\x00\x00\x00\x10\x50\x6B\x26\x37\x63\xD7\x82\x31\x55\x2E'
 b'\x3B\xB5\x81\xB5\x33\x3A\x9E\x53\x23\x3C\x31\x2E\xA9\xCC\xF5\xAB\x00\x00\x00\x00\x00\x10\xCF\xE7\x41\xE1\x6E\xE9\x44\xCC\x06\x9B'
 b'\x55\x52\x32\xF0\x15\x42\x92\x7F\xE9\xB6\x11\x07\xCD\xA6\x35\x4E\x00\x00\x00\x00\x00\x10\x90\x64\xA5\x37\x55\xB6\xA0\xC7\xD8\xB0'
 b'\x73\x1C\xCA\x14\x74\xC4\x60\x12\xA6\x6E\x35\x80\x5A\xE3\xAD\xF7\x00\x00\x00\x00\x00\x10\x80\x7F\x9F\xBB\xAA\x94\x82\x5A\x2B\xB1'
 b'\xE4\x2A\xDA\x83\x58\xCB\x2D\xBA\xCD\x74\x89\xE4\x0D\x7C\x7E\xB0\x00\x00\x00\x00\x00\x10\xD1\x6C\xD6\xB3\xB3\x28\x9B\x41\xB6\x2C'
 b'\x9E\xBB\x33\xB7\x4C\x8C\x1A\xA5\x43\x5B\x9D\x4C\x8E\xA4\xCC\x18\x

In [9]:
type(data[0])

numpy.void

## logger

In [ ]:
from cracknuts import logger
import logging

logger.set_level(logging.DEBUG, cracker)

## temp

In [ ]:
import random

In [ ]:
cmd_set_key = '02 00 00 00 00 00 00 08'
cmd_set_key = cmd_set_key.replace(' ', '')
cmd_set_key = bytes.fromhex(cmd_set_key)
print(cmd_set_key)
len(cmd_set_key)
key = random.randbytes(8)
cmd_set_key = cmd_set_key + key
print(cmd_set_key)
len(cmd_set_key)

In [ ]:
random.randbytes(8)

In [ ]:
t = np.frombuffer(cmd_set_key)

In [ ]:
t = cmd_set_key

In [ ]:
print(t)

In [ ]:
type(cmd_set_key)

In [ ]:
cmd_set_key[-8:]